# 01 · Your first LLM call

In this notebook we:

1. call a large language model (LLM) from Python,
2. look at the raw response we get back,
3. understand what happens behind the API call.

**A note on the `openai` package.** It is a client for the *chat completions* API, a JSON format that
many providers speak, not just OpenAI. Google's Gemini (free tier), Groq (free tier) and Ollama (runs on
your laptop) all implement it, so the same code works with any of them. Which provider and model we use is
decided in the `.env` file, not in the code.

In [7]:
import os

from dotenv import load_dotenv
from openai import OpenAI

# Read LLM_BASE_URL, LLM_API_KEY and LLM_MODEL from the .env file into environment variables.
# (.env is git-ignored, so the key never ends up in the repository.)
load_dotenv()

# The client only needs to know WHERE to send requests and WHICH key to use.
client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
MODEL = os.environ["LLM_MODEL"]

print("Using model:", MODEL)

Using model: gemini-3.8-flash


## 1. One chat completion

A request is a **list of messages**, each with a *role*:

| role | who | used for |
|---|---|---|
| `system` | us | instructions that frame the whole conversation (persona, rules) |
| `user` | the end user | the question or task |
| `assistant` | the model | its previous answers, when we continue a conversation |

The model reads the whole list and produces the next `assistant` message.

In [2]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a marketing analyst."},
        {"role": "user", "content": "In two sentences, what is customer sentiment analysis?"},
    ],
)

# The answer lives in the first (and usually only) "choice".
print(response.choices[0].message.content)

Customer sentiment analysis is the process of using natural language processing and text analytics to identify the emotional tone—positive, negative, or neutral—behind customer feedback, reviews, and online conversations. It allows marketers to quantify consumer perceptions at scale, enabling data-driven decisions that improve campaign performance, protect brand reputation, and enhance the overall customer experience.


## 2. What happens behind the API call

The HTTP request above triggers a fixed pipeline on the server. Nothing in it is magic:

![The LLM inference pipeline](img/llm_pipeline.svg)

1. **Chat template.** The list of messages is rendered into *one* string. Special tokens mark where each
   role starts and ends, e.g. `<|start|>user<|message|>In two sentences, …<|end|>`. Every model
   family has its own template; the server applies the right one for us. This is why the prompt is
   "bigger" than the words we typed (see `prompt_tokens` below).

2. **Tokenisation.** The string is cut into *tokens* (whole words, pieces of words, punctuation) and each
   token is replaced by an integer id from the model's vocabulary. The model never sees letters, only ids.

3. **Forward pass.** Each id is looked up in an embedding table and becomes a vector; the sequence of
   vectors goes through the transformer layers. The model produces an output at *every* position, but the
   one we care about is at the **last position**: a probability distribution over the whole vocabulary,
   i.e. "what token comes next?".

4. **Generation loop.** One token is chosen from that distribution (the most likely one, or a random draw
   weighted by probability, which is what `temperature` controls). It is appended to the sequence and the
   model runs again. In practice the server keeps the intermediate results of the earlier tokens (the
   *KV cache*), so each iteration only processes the one new token. One iteration = one new token. The loop
   stops when the model emits a special *stop* token or when `max_tokens` is reached. `completion_tokens`
   counts these iterations.
   Some models (the Gemini 3 family, gpt-oss, …) first generate hidden *reasoning* tokens ("I need to
   answer in two sentences…") before the visible answer. Providers differ in what they show of them: some
   return the text, some only count them in `reasoning_tokens`, some show nothing.

5. **Detokenisation.** The generated ids are mapped back to text. The special tokens are stripped and used
   to split the output: the answer becomes `content`, and hidden reasoning, if the provider exposes it,
   becomes a separate field. The result is wrapped in the JSON we receive.

**Padding.** A server does not process one request at a time: it batches many of them. In the simplest
form of batching the sequences are stacked into one tensor, so they must have the same length; shorter
ones are extended with a special *pad* token that is masked out of the attention computation. For
generation the padding goes on the *left*, so that every sequence's last real token sits in the same
column and the "next token" is read from the same place for all of them. Modern inference servers use
*continuous batching*, which packs sequences of different lengths together and needs little or no padding,
but the principle is the same: padding is a mechanical requirement of batching and never changes the answer.

Let's look at steps 2 and 5 ourselves. Gemini's tokeniser is not published as a library, so we use
`tiktoken`, the tokeniser of OpenAI's models, including the open gpt-oss models you can run on Groq.
Every model family has its own vocabulary, but the mechanics are the same.

In [3]:
import tiktoken

# "o200k_base" is the vocabulary family used by the gpt-oss models (other models have their own).
enc = tiktoken.get_encoding("o200k_base")

text = "In two sentences, what is customer sentiment analysis?"

ids = enc.encode(text)  # step 2: text -> token ids
print("ids:   ", ids)

# Decode each id on its own to see where the token boundaries are.
# Note the leading spaces: " two" and "two" are different tokens.
print("tokens:", [enc.decode([i]) for i in ids])

print(f"{len(text.split())} words became {len(ids)} tokens")

print("decoded:", enc.decode(ids))  # step 5: token ids -> text

ids:    [637, 1920, 40536, 11, 1412, 382, 5989, 39160, 8450, 30]
tokens: ['In', ' two', ' sentences', ',', ' what', ' is', ' customer', ' sentiment', ' analysis', '?']
8 words became 10 tokens
decoded: In two sentences, what is customer sentiment analysis?


### The chat template, made visible

Step 1 is normally hidden inside the server, and closed models such as Gemini do not publish their
template at all. The open gpt-oss family does, so we use it as the example. Below is what our two messages
look like once its *harmony* template has been applied: one string, with special tokens such as
`<|start|>`, `<|message|>` and `<|end|>` marking the structure. Two things to notice:

* the model has its own `system` header (identity, cutoff date, reasoning effort), and **our** "system"
  message is placed in a `developer` block under `# Instructions`;
* the string ends with `<|start|>assistant`, an open block: the model's job is simply to continue it.

Each special token is a *single* id in the vocabulary. By default `tiktoken` refuses to encode them, so
that text coming from a user can never contain a control token by accident (or on purpose). We opt in here
because we are building the template ourselves.

In [4]:
# The rendered template for our two messages (written out by hand so it is visible here).
rendered = (
    "<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.\n"
    "Knowledge cutoff: 2024-06\n\n"
    "Reasoning: medium\n\n"
    "# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|>"
    "<|start|>developer<|message|># Instructions\n\n"
    "You are a marketing analyst.<|end|>"
    "<|start|>user<|message|>In two sentences, what is customer sentiment analysis?<|end|>"
    "<|start|>assistant"
)
print(rendered)

# "o200k_harmony" = the o200k vocabulary plus the harmony special tokens.
enc_h = tiktoken.get_encoding("o200k_harmony")
ids = enc_h.encode(rendered, allowed_special="all")  # special tokens are off by default: opt in

print("\n", len(ids), "tokens in total. A provider usually adds a few lines of its own (e.g. the date),"
      " and each model counts with its own vocabulary, so usage.prompt_tokens will differ.")

# The first few tokens, one per line: the special tokens are single ids.
for i in ids[:8]:
    print(f"{i:>7}  {enc_h.decode([i])!r}")

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>developer<|message|># Instructions

You are a marketing analyst.<|end|><|start|>user<|message|>In two sentences, what is customer sentiment analysis?<|end|><|start|>assistant

 79 tokens in total. A provider usually adds a few lines of its own (e.g. the date), and each model counts with its own vocabulary, so usage.prompt_tokens will differ.
 200006  '<|start|>'
  17360  'system'
 200008  '<|message|>'
   3575  'You'
    553  ' are'
  17554  ' Chat'
 162016  'GPT'
     11  ','


## 3. The raw response

Printing `.content` hides most of what came back. The full object tells us *how* the answer was produced.

In [5]:
# .model_dump() turns the response object into a plain dictionary.
# Things to notice:
#   choices[0].finish_reason  -> "stop" means the model ended by itself (not cut off by max_tokens)
#   usage.prompt_tokens       -> size of our messages AFTER the chat template was applied
#   usage.completion_tokens   -> how many generation-loop iterations ran (reasoning + answer)
#   Some providers add fields of their own, e.g. Groq returns the hidden reasoning text as
#   choices[0].message.reasoning. The core fields are the same everywhere.
response.model_dump()

{'id': 'yD_map2QEYWgxN8PzdPUuQM',
 'choices': [{'finish_reason': 'stop',
   'index': 0,
   'logprobs': None,
   'message': {'content': 'Customer sentiment analysis is the process of using natural language processing and text analytics to identify the emotional tone—positive, negative, or neutral—behind customer feedback, reviews, and online conversations. It allows marketers to quantify consumer perceptions at scale, enabling data-driven decisions that improve campaign performance, protect brand reputation, and enhance the overall customer experience.',
    'refusal': None,
    'role': 'assistant',
    'annotations': None,
    'audio': None,
    'function_call': None,
    'tool_calls': None,
    'extra_content': {'google': {'thought_signature': 'EukSCuYSAWkUfROhMJwZQ0hY6YRtKudXnsLw9k8Fn+VKfDvto2+f/3Arx4TeAoFy6uXDpF5X40/RzhNEkzTC0uURjhrr05Z5dN7pJOA60AV5u0LFAtk/5rTKI7b5UWPYhybwLYONAdIoLh90ClB/Y7wt+3cRXWEjnzVHc79S3fSoqvHjYJMxKa6VUI1p3Fw3XyI2EifhuG3EIQadA+ovCWiBHkQt/E0/w/wSi+9i+W7cOPDU10Cm

In [6]:
# The same numbers, pulled out explicitly. Free tiers limit tokens per minute, so these matter.
usage = response.usage
print("prompt tokens:    ", usage.prompt_tokens)
print("completion tokens:", usage.completion_tokens)

# Not every provider reports reasoning tokens separately, so check before printing.
details = usage.completion_tokens_details
if details is not None and details.reasoning_tokens is not None:
    print("  of which reasoning:", details.reasoning_tokens)

prompt tokens:     18
completion tokens: 68


## 4. Switching provider

Nothing above mentions Gemini. `.env.example` lists two alternatives:

* **Groq**, another free hosted provider, very fast, with a much tighter tokens-per-minute limit;
* **Ollama**, a model running on your own machine: install [Ollama](https://ollama.com) and run
  `ollama pull llama3.1:8b`.

To switch: in `.env`, comment out the Gemini lines, uncomment the ones you want, restart the kernel and run
the notebook again. The only thing that changed is `base_url`: the client now talks to a different server.
Local models are smaller and slower, but the API is the same.

One caveat: the chat template and the tokeniser shown in section 2 belong to the gpt-oss family. Each
model has its own template (Llama's starts with `<|begin_of_text|><|start_header_id|>…`) and its own
vocabulary, so the token counts differ. The five steps are the same for every model.